In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce

DATA_PROCESSED = Path("data/processed")
FIG_DIR = Path("figures")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2013
END_YEAR = 2024

EA_COUNTRIES = [
    "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Estonia",
    "Finland", "France", "Germany", "Greece", "Ireland", "Italy",
    "Latvia", "Lithuania", "Luxembourg", "Malta", "Netherlands",
    "Portugal", "Slovakia", "Slovenia", "Spain"
]

In [2]:
def basic_clean(df, geo_col="geo", time_col="TIME_PERIOD", value_col="OBS_VALUE"):
    """Standardise column names, filter years and euro area countries."""
    df = df.rename(columns=lambda c: c.strip())
    assert geo_col in df.columns and time_col in df.columns and value_col in df.columns

    # Keep relevant columns, but don't drop others yet
    df = df[[geo_col, time_col, value_col] + [c for c in df.columns if c not in [geo_col, time_col, value_col]]]

    # Keep only yearly observations (TIME_PERIOD like 2013, not 2013Q1)
    df[time_col] = df[time_col].astype(str)
    df = df[df[time_col].str.len() == 4]

    df[time_col] = pd.to_numeric(df[time_col], errors="coerce")
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df = df.dropna(subset=[time_col, value_col]).copy()

    df[time_col] = df[time_col].astype(int)
    df = df[(df[time_col] >= START_YEAR) & (df[time_col] <= END_YEAR)]

    df = df[df[geo_col].isin(EA_COUNTRIES)].copy()

    df = df.rename(columns={geo_col: "country", time_col: "year", value_col: "value"})
    return df

In [3]:
birth_raw = pd.read_csv("DATA_RAW/birth.csv", low_memory=False)
birth_crude = birth_raw[birth_raw["indic_de"] == "Crude birth rate - per thousand persons"].copy()
birth = basic_clean(birth_crude)
birth = birth[["country", "year", "value"]].rename(columns={"value": "birth_rate"})

print("Birth:", birth.shape)
birth.head()

Birth: (252, 3)


,country,year,birth_rate
25,Austria,2013,9.4
26,Austria,2014,9.6
27,Austria,2015,9.8
28,Austria,2016,10.0
29,Austria,2017,10.0


In [4]:
mar_raw = pd.read_csv("DATA_RAW/marriage.csv", low_memory=False)

mar_sel = mar_raw[
    mar_raw["indic_de"].isin([
        "Mean age at first marriage - females",
        "Mean age at first marriage - males"
    ])
].copy()

mar = basic_clean(mar_sel)
mar_grouped = (
    mar.groupby(["country", "year"], as_index=False)["value"]
       .mean()
       .rename(columns={"value": "mean_age_first_marriage"})
)

print("Marriage:", mar_grouped.shape)
mar_grouped.head()

Marriage: (190, 3)


,country,year,mean_age_first_marriage
0,Austria,2013,32.00
1,Austria,2014,32.25
2,Austria,2015,32.55
3,Austria,2016,32.65
4,Austria,2017,32.85


In [5]:
hpi_raw = pd.read_csv("DATA_RAW/hpi.csv", low_memory=False)

hpi_sel = hpi_raw[
    (hpi_raw["purchase"] == "Total") &
    (hpi_raw["unit"] == "Annual average index, 2015=100")
].copy()

hpi = basic_clean(hpi_sel, geo_col="geo", time_col="TIME_PERIOD", value_col="OBS_VALUE")
hpi = hpi[["country", "year", "value"]].rename(columns={"value": "hpi_index"})

print("HPI:", hpi.shape)
hpi.head()


HPI: (240, 3)


,country,year,hpi_index
2983,Austria,2013,91.20
2984,Austria,2014,94.68
2985,Austria,2015,100.00
2986,Austria,2016,106.71
2987,Austria,2017,112.14


In [6]:
hicp_raw = pd.read_csv("DATA_RAW/hicp.csv", low_memory=False)
hicp_sel = hicp_raw[
    (hicp_raw["coicop"] == "All-items HICP") &
    (hicp_raw["unit"] == "Annual average index")
].copy()
hicp = basic_clean(hicp_sel, geo_col="geo", time_col="TIME_PERIOD", value_col="OBS_VALUE")
hicp = hicp[["country", "year", "value"]].rename(columns={"value": "hicp_index"})

print("HICP:", hicp.shape)
hicp.head()


HICP: (252, 3)


,country,year,hicp_index
9,Austria,2013,97.77
10,Austria,2014,99.20
11,Austria,2015,100.00
12,Austria,2016,100.97
13,Austria,2017,103.22


In [7]:
gdp_raw = pd.read_csv("DATA_RAW/gdp.csv", low_memory=False)

gdp_sel = gdp_raw[
    gdp_raw["unit"] == "Chain linked volumes, percentage change on previous period"
].copy()

gdp = basic_clean(gdp_sel, geo_col="geo", time_col="TIME_PERIOD", value_col="OBS_VALUE")
gdp = gdp[["country", "year", "value"]].rename(columns={"value": "gdp_growth"})

print("GDP:", gdp.shape)
gdp.head()


GDP: (231, 3)


,country,year,gdp_growth
11,Austria,2014,0.8
12,Austria,2015,1.3
13,Austria,2016,2.1
14,Austria,2017,2.3
15,Austria,2018,2.5


In [8]:
unemp_raw = pd.read_csv("DATA_RAW/unemp_cit.csv", low_memory=False)

unemp_sel = unemp_raw[
    (unemp_raw["age"] == "From 15 to 74 years") &
    (unemp_raw["citizen"] == "Total") &
    (unemp_raw["unit"] == "Percentage")
].copy()

unemp = basic_clean(unemp_sel, geo_col="geo", time_col="TIME_PERIOD", value_col="OBS_VALUE")

unemp_grouped = (
    unemp.groupby(["country", "year"], as_index=False)["value"]
         .mean()
         .rename(columns={"value": "unemp_rate"})
)

print("Unemployment:", unemp_grouped.shape)
unemp_grouped.head()


Unemployment: (252, 3)


,country,year,unemp_rate
0,Austria,2013,5.366667
1,Austria,2014,5.633333
2,Austria,2015,5.700000
3,Austria,2016,6.033333
4,Austria,2017,5.466667


In [9]:
dfs = [birth, mar_grouped, hpi, hicp, gdp, unemp_grouped]

panel = reduce(
    lambda left, right: pd.merge(left, right, on=["country", "year"], how="inner"),
    dfs
)

panel = panel.sort_values(["country", "year"]).reset_index(drop=True)

print("Panel shape:", panel.shape)
panel.head()


Panel shape: (163, 8)


,country,year,birth_rate,mean_age_first_marriage,hpi_index,hicp_index,gdp_growth,unemp_rate
0,Austria,2014,9.6,32.25,94.68,99.20,0.8,5.633333
1,Austria,2015,9.8,32.55,100.00,100.00,1.3,5.700000
2,Austria,2016,10.0,32.65,106.71,100.97,2.1,6.033333
3,Austria,2017,10.0,32.85,112.14,103.22,2.3,5.466667
4,Austria,2018,9.7,33.00,118.83,105.41,2.5,4.866667


In [10]:
# Growth rates for price indices (year-on-year %)
panel["hpi_growth"] = panel.groupby("country")["hpi_index"].pct_change() * 100
panel["hicp_growth"] = panel.groupby("country")["hicp_index"].pct_change() * 100

# Lags (1-year) for macro variables
for col in ["hpi_growth", "hicp_growth", "gdp_growth", "unemp_rate"]:
    panel[f"{col}_lag1"] = panel.groupby("country")[col].shift(1)

panel_clean = panel.dropna().reset_index(drop=True)
panel_clean.to_csv(DATA_PROCESSED / "ea_country_panel_clean.csv", index=False)

panel_clean.head(), panel_clean.shape


(   country  year  birth_rate  mean_age_first_marriage  hpi_index  hicp_index  \
 0  Austria  2016        10.0                    32.65     106.71      100.97   
 1  Austria  2017        10.0                    32.85     112.14      103.22   
 2  Austria  2018         9.7                    33.00     118.83      105.41   
 3  Austria  2019         9.6                    33.35     125.97      106.98   
 4  Austria  2023         8.5                    34.00     163.68      130.40   
 
    gdp_growth  unemp_rate  hpi_growth  hicp_growth  hpi_growth_lag1  \
 0         2.1    6.033333    6.710000     0.970000         5.618927   
 1         2.3    5.466667    5.088558     2.228385         6.710000   
 2         2.5    4.866667    5.965757     2.121682         5.088558   
 3         1.8    4.500000    6.008584     1.489422         5.965757   
 4        -0.8    5.100000   29.935699    21.891942         6.008584   
 
    hicp_growth_lag1  gdp_growth_lag1  unemp_rate_lag1  
 0          0.806452 

In [11]:
panel_clean.describe(include="all")
panel_clean.isna().mean().sort_values(ascending=False)

country                    0.0
year                       0.0
birth_rate                 0.0
mean_age_first_marriage    0.0
hpi_index                  0.0
hicp_index                 0.0
gdp_growth                 0.0
unemp_rate                 0.0
hpi_growth                 0.0
hicp_growth                0.0
hpi_growth_lag1            0.0
hicp_growth_lag1           0.0
gdp_growth_lag1            0.0
unemp_rate_lag1            0.0
dtype: float64

In [12]:
mean_by_year = (
    panel_clean.groupby("year")[[
        "birth_rate", "mean_age_first_marriage",
        "hpi_growth", "hicp_growth",
        "gdp_growth", "unemp_rate"
    ]]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 5))
sns.lineplot(data=mean_by_year, x="year", y="birth_rate", label="Birth rate")
plt.ylabel("Per 1,000 persons")
plt.title("Average birth rate – euro area countries")
plt.tight_layout()
plt.savefig(FIG_DIR / "ts_birth_rate.png", dpi=200)
plt.close()

plt.figure(figsize=(10, 5))
sns.lineplot(data=mean_by_year, x="year", y="mean_age_first_marriage", label="Mean age at first marriage")
plt.ylabel("Years")
plt.title("Average mean age at first marriage – euro area countries")
plt.tight_layout()
plt.savefig(FIG_DIR / "ts_mean_age_marriage.png", dpi=200)
plt.close()

plt.figure(figsize=(10, 5))
sns.lineplot(data=mean_by_year, x="year", y="hpi_growth", label="HPI growth")
sns.lineplot(data=mean_by_year, x="year", y="hicp_growth", label="HICP growth")
plt.ylabel("Percent")
plt.title("Average housing price and inflation growth – euro area")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "ts_price_growth.png", dpi=200)
plt.close()


In [13]:
# Birth vs lagged housing price growth
sns.lmplot(
    data=panel_clean,
    x="hpi_growth_lag1",
    y="birth_rate",
    height=5,
    aspect=1.3,
    scatter_kws={"alpha": 0.4}
)
plt.title("Birth rate vs lagged HPI growth")
plt.tight_layout()
plt.savefig(FIG_DIR / "birth_vs_hpi_lag1.png", dpi=200)
plt.close()

# Birth vs lagged inflation
sns.lmplot(
    data=panel_clean,
    x="hicp_growth_lag1",
    y="birth_rate",
    height=5,
    aspect=1.3,
    scatter_kws={"alpha": 0.4}
)
plt.title("Birth rate vs lagged HICP growth")
plt.tight_layout()
plt.savefig(FIG_DIR / "birth_vs_hicp_lag1.png", dpi=200)
plt.close()

# Mean age at first marriage vs lagged HPI growth
sns.lmplot(
    data=panel_clean,
    x="hpi_growth_lag1",
    y="mean_age_first_marriage",
    height=5,
    aspect=1.3,
    scatter_kws={"alpha": 0.4}
)
plt.title("Mean age at first marriage vs lagged HPI growth")
plt.tight_layout()
plt.savefig(FIG_DIR / "marriage_age_vs_hpi_lag1.png", dpi=200)
plt.close()


In [14]:
corr_cols = [
    "birth_rate", "mean_age_first_marriage",
    "hpi_growth", "hpi_growth_lag1",
    "hicp_growth", "hicp_growth_lag1",
    "gdp_growth", "gdp_growth_lag1",
    "unemp_rate", "unemp_rate_lag1"
]

corr = panel_clean[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0)
plt.title("Correlation matrix: demography & macro variables (EA countries)")
plt.tight_layout()
plt.savefig(FIG_DIR / "corr_panel.png", dpi=200)
plt.close()
